# 01 · Explore the map

Get to know the reference raster before writing any matching code.

`data/dresden-map.png` is 8192 x 5074 (41.6 MPix), **RGBA** — `cv2.imread` drops alpha by default, which is usually what you want, but be deliberate about it.

In [ ]:
%load_ext autoreload
%autoreload 2

import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from navmod import matcher, camera, metrics, georef

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"


def show(img, title=None, figsize=(11, 7)):
    """Inline display for OpenCV arrays (handles BGR->RGB and grayscale)."""
    plt.figure(figsize=figsize)
    gray = img.ndim == 2
    plt.imshow(img if gray else cv2.cvtColor(img, cv2.COLOR_BGR2RGB),
               cmap="gray" if gray else None)
    if title:
        plt.title(title)
    plt.axis("off")
    plt.show()


print("ready ·", cv2.__version__)

## Load it

Check dtype, shape and channel count. Decide once whether you work in colour or grayscale and stay consistent — detectors want grayscale anyway.

## Establish scale

What ground distance does the map span? That fixes your GSD (metres/pixel) and makes every later error number physically meaningful. Good first use of `metrics.gsd`.

If it is 5 km across 8192 px, GSD is about 0.61 m/px.

## Look at crops

Pull a few 512x512 windows from different regions — dense city, river, parkland, farmland. These are what your drone will actually see.

## Where are the features?

Run a detector over several crops and count keypoints. Feature-poor regions (water, uniform fields) are where localization will fail, and knowing *which* parts of your map are weak will explain later results. Consider a coarse heatmap of keypoint density over the whole map.